
# 練習問題: モンテカルロ法で π を求める (reduction)

## 目標

`reduction(+:count)` 句を1つ追加するだけで, 複数のスレッドが同じカウンタ `count` を同時に増やそうとして起こる競合 (data race) を解消し,
モンテカルロ法による円周率 π の推定を正しく並列化できることを体験する.

## 課題

`montecarlo_pi.cpp` (または `montecarlo_pi.f90`) は, 単位正方形 [0,1)×[0,1) に n 個の点をランダムに投げ,
原点からの距離が 1 未満 (= 半径 1 の円の 1/4 の内側) に入った点の数 `count` を数えて, π ≈ 4 * count / n を求めるコードである.
乱数は反復番号から決まる線形合同法 (LCG) で生成しているので, スレッド数によらず同じ点列になる.

このループを単純に並列化すると, 全スレッドが共有変数 `count` に同時に `count++` を行うため競合 (data race) が起き,
スレッド数が2以上だと**数え落とし**が発生して π が小さめに出たり, 実行ごとに値が変わったりする.

コメント `TODO` の箇所で **`reduction` を用いて並列化せよ**. 各スレッドが部分カウントを持ち寄って正しい合計を得るようになる.

- C++: `#pragma omp parallel for reduction(+:count)`
- Fortran: `!$omp parallel do private(x, y) reduction(+:count)` … `!$omp end parallel do`

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore montecarlo_pi.cpp -o montecarlo_pi.exe

# Fortran
nvfortran -fast -mp=multicore montecarlo_pi.f90 -o montecarlo_pi.exe
```

```
OMP_NUM_THREADS=4 ./montecarlo_pi.exe
```

第1引数で点数 `n` を変えられる (例: `./montecarlo_pi.exe 1000000000`).

## 期待される結果

点数 n を大きくするほど, π の推定値は真の値 3.14159... に近づく.

```
pi ~= 3.141...
```

- `reduction(+:count)` を**追加すると**, スレッド数によらず常に同じ正しい値になる.
- 参考: もし `reduction` を付けずに共有変数 `count` をそのまま `count++` で並列に更新すると (例: `OMP_NUM_THREADS=4`),
  複数スレッドの加算がぶつかって数え落としが起き, `count` が本来より小さくなって π が 3.14 より小さく出たり, 実行ごとに値が変わったりする.
  これが reduction が必要な理由である.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit_a` (Aquarius) などでジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ montecarlo_pi.cpp
#include <cstdint>
#include <cstdio>
#include <cstdlib>

/* 反復番号 i から決まる擬似乱数 (splitmix64) を [0,1) の double で返す.
   毎回 i から計算するので, スレッドごとの状態を持たず並列化しやすい. */
static inline double lcg01(uint64_t i) {
  uint64_t x = (i + 1) * 6364136223846793005ULL + 1442695040888963407ULL;
  /* よくかき混ぜる */
  x ^= x >> 30; x *= 0xbf58476d1ce4e5b9ULL;
  x ^= x >> 27; x *= 0x94d049bb133111ebULL;
  x ^= x >> 31;
  /* 上位 53 bit を [0,1) の double に */
  return (double)(x >> 11) * (1.0 / 9007199254740992.0);
}

int main(int argc, char ** argv) {
  long n = (1 < argc ? atol(argv[1]) : 100L * 1000L * 1000L);
  long count = 0;               /* 単位円の 1/4 の内側に入った点数 */
  printf("n = %ld\n", n);
  /* 単位正方形 [0,1)x[0,1) に n 点を投げ, 半径 1 の円の内側に入った点を数える. */
  // TODO: 円内に入った点数を reduction(+:count) で集計して π を求めよ.
  for (long i = 0; i < n; i++) {
    double x = lcg01(2 * i);
    double y = lcg01(2 * i + 1);
    if (x * x + y * y < 1.0) count++;
  }
  double pi = 4.0 * (double)count / (double)n;
  printf("count = %ld / %ld\n", count, n);
  printf("pi ~= %.6f\n", pi);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore montecarlo_pi.cpp -o montecarlo_pi_cpp.exe

## 2-2. 実行
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./montecarlo_pi_cpp.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a

./montecarlo_pi_cpp.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./montecarlo_pi_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ montecarlo_pi.f90
module lcg_mod
contains
  ! a * b の下位 64bit を, 符号付き整数の overflow (Fortran では未定義動作) を
  ! 避けるため 32bit ずつに分割して計算する.
  function mulw(a, b) result(p)
    integer(8), intent(in) :: a, b
    integer(8) :: p, al, ah, bl, bh, mid
    integer(8), parameter :: mask = 4294967295_8     ! 2^32 - 1
    al = iand(a, mask); ah = ishft(a, -32)
    bl = iand(b, mask); bh = ishft(b, -32)
    mid = iand(al * bh + ah * bl, mask)   ! さらに上位は << 32 で消えるので捨てる
    p = al * bl + ishft(mid, 32)
  end function mulw

  ! 反復番号 idx から決まる擬似乱数 (splitmix64) を [0,1) の double で返す.
  ! 毎回 idx から計算するので, スレッドごとの状態を持たず並列化しやすい.
  function lcg01(idx) result(r)
    integer(8), intent(in) :: idx
    integer(8) :: x
    real(8) :: r
    x = idx + 1
    x = mulw(x, 6364136223846793005_8) + 1442695040888963407_8
    x = ieor(x, ishft(x, -30)); x = mulw(x, -4658895280553007687_8)
    x = ieor(x, ishft(x, -27)); x = mulw(x, -7723592293110705685_8)
    x = ieor(x, ishft(x, -31))
    ! 上位 53 bit を [0,1) の double に
    r = real(ishft(x, -11), 8) * (1.0d0 / 9007199254740992.0d0)
  end function lcg01
end module lcg_mod

program montecarlo_pi
  use lcg_mod
  character(len=64) :: arg
  integer(8) :: n, i, count
  real(8) :: x, y, pi
  n = 100_8 * 1000_8 * 1000_8
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  count = 0                      ! 単位円の 1/4 の内側に入った点数
  print "(a,i0)", "n = ", n
  ! 単位正方形 [0,1)x[0,1) に n 点を投げ, 半径 1 の円の内側に入った点を数える.
  ! TODO: 円内に入った点数を reduction(+:count) で集計して π を求めよ.
  do i = 0, n - 1
     x = lcg01(2 * i)
     y = lcg01(2 * i + 1)
     if (x * x + y * y < 1.0d0) count = count + 1
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
  pi = 4.0d0 * real(count, 8) / real(n, 8)
  print "(a,i0,a,i0)", "count = ", count, " / ", n
  print "(a,f0.6)", "pi ~= ", pi
end program montecarlo_pi

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore montecarlo_pi.f90 -o montecarlo_pi_f90.exe

## 3-2. 実行
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./montecarlo_pi_f90.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a
./montecarlo_pi_f90.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit

#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./montecarlo_pi_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:montecarlo_pi.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:montecarlo_pi.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:montecarlo_pi.cpp}